# UR3e Reach — Setup, Simulation Walkthrough & Reward Tuning

This notebook walks through:

1. Picking the right kernel (this folder has no Python env of its own)
2. Registering and building the `Mjlab-UR3e-Reach` task
3. Resetting the environment and looking at it (offscreen render)
4. Running a short rollout
5. **The reward deep-dive** — inspecting each reward term live, and changing/adding
   terms *without* editing `ur3e_reach_env_cfg.py` every time you want to try something

This notebook lives inside the `ur3e/` package itself
([`ur3e_reach_env_cfg.py`](ur3e_reach_env_cfg.py), [`commands.py`](commands.py)) and
imports it directly — it never touches the other copy of this task under
`my_mjlab_project`, so anything you do here only affects this folder.

## 0. Kernel

This notebook runs on **`MyCode/.venv`** — the shared virtual environment for
everything under `MyCode/` (also used by `FallingBall` and `pendulum`). `mjlab`,
`torch`, and `mujoco-warp` are installed there, so `ur3e/` runs standalone with no
dependency on the `hackathon/my_mjlab_project` folder or its own venv.

Select the **`mjlab (UR3e reach)`** kernel from the kernel picker (top-right of this
notebook) before running anything below.

`MyCode/` now has its own `pyproject.toml` + `uv.lock` recording exactly what's
installed. If the venv is ever wiped or you're on a new machine, rebuild it with:

```bash
cd /Users/crogers/GitHub/MyCode
uv sync
.venv/bin/python -m ipykernel install --user --name mjlab-ur3e --display-name "mjlab (UR3e reach)"
```

**Version note:** `mjlab==1.5.0` is pinned in `pyproject.toml` alongside
`mujoco==3.10.0`, `mujoco-warp==3.10.0.1`, `warp-lang==1.14.0`, and `torch==2.12.1`.
Newer patch releases of `mujoco-warp`/`warp-lang` (3.10.0.3 / 1.15.0, as of writing)
have a regression that breaks multi-env `reset()` -- if you ever bump these packages
(e.g. via `uv add --upgrade-package`) and cell 6 below throws `IndexError: index N is
out of bounds for dimension 0 with size 1`, that's why; pin back to the versions
above.

In [ ]:
import sys
from pathlib import Path

# Jupyter's default cwd is this notebook's own folder (`ur3e/`). Add its *parent*
# to sys.path so `import ur3e` resolves to this local copy of the task package,
# not some other copy elsewhere on disk.
UR3E_DIR = Path.cwd()
if UR3E_DIR.name != "ur3e":
    UR3E_DIR = Path("/Users/crogers/GitHub/MyCode/ur3e")
MYCODE_DIR = UR3E_DIR.parent
if str(MYCODE_DIR) not in sys.path:
    sys.path.insert(0, str(MYCODE_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt

import ur3e  # noqa: F401 -- side effect: registers Mjlab-UR3e-Reach(+ClampedPlay)
from mjlab.tasks.registry import list_tasks, load_env_cfg
from mjlab.envs import ManagerBasedRlEnv

ur3e_tasks = [t for t in list_tasks() if "UR3e" in t]
print("Registered UR3e tasks:", ur3e_tasks)

## 1. What this task is

- **Action** (6D): relative joint-position deltas for the 6 arm joints —
  `target = current_qpos + action * 0.08 rad`
  ([`ur3e_reach_env_cfg.py:433`](ur3e_reach_env_cfg.py#L433)).
- **Observation**: joint pos/vel (relative to default pose), the commanded target
  position, and the end-effector→target vector
  ([`ur3e_reach_env_cfg.py:348`](ur3e_reach_env_cfg.py#L348)).
- **Command**: a randomly sampled reach target in a hemisphere in front of the base,
  resampled every 9–12s ([`commands.py`](commands.py),
  [`ur3e_reach_env_cfg.py:154`](ur3e_reach_env_cfg.py#L154)).
- **Reward**: distance-to-target shaping + a success bonus + several safety/smoothness
  penalties — the whole point of section 5 below.
- **Termination**: time-out only (no early termination on collision — collisions are
  just penalized in the reward).

In [ ]:
cfg = load_env_cfg("Mjlab-UR3e-Reach")
print(f"num_envs (train default): {cfg.scene.num_envs}")
step_dt = cfg.decimation * cfg.sim.mujoco.timestep
print(f"decimation: {cfg.decimation}, physics dt: {cfg.sim.mujoco.timestep}s -> step dt: {step_dt}s")
print(f"episode length: {cfg.episode_length_s}s ({round(cfg.episode_length_s / step_dt)} steps)")
print(f"action terms: {list(cfg.actions.keys())}")
print(f"observation groups: {list(cfg.observations.keys())}")
print(f"reward terms: {list(cfg.rewards.keys())}")

## 2. Build a small environment and reset it

`num_envs=512` is the training default (sized for GPU throughput). For interactive
use, override it to something small — this also runs fine on CPU.

In [ ]:
cfg = load_env_cfg("Mjlab-UR3e-Reach")
cfg.scene.num_envs = 4
env = ManagerBasedRlEnv(cfg, device="cpu")

obs, extras = env.reset()
print("obs groups:", list(obs.keys()))
print("actor obs shape:", obs["actor"].shape)
print("action dim:", env.action_manager.total_action_dim)

## 3. Look at it

The interactive MuJoCo viewer (`uv run play`) needs a real window and won't work
embedded in a notebook, but we can grab an offscreen RGB frame and show it inline —
enough to sanity-check the scene, the arm's home pose, and the sampled target
(green sphere).

In [ ]:
%matplotlib inline

play_cfg = load_env_cfg("Mjlab-UR3e-Reach", play=True)
play_cfg.scene.num_envs = 1
render_env = ManagerBasedRlEnv(play_cfg, device="cpu", render_mode="rgb_array")
render_env.reset()

frame = render_env.render()
plt.figure(figsize=(5, 4))
plt.imshow(frame)
plt.axis("off")
plt.title("UR3e reach -- reset state (green sphere = target)")
plt.show()

## 4. Run a short rollout

Random actions, just to see reset → step → reward/termination handling work end to
end before diving into the reward internals.

In [ ]:
NUM_STEPS = 100
action_dim = env.action_manager.total_action_dim

total_reward_history = []
for _ in range(NUM_STEPS):
    action = torch.randn(env.num_envs, action_dim) * 0.3
    obs, reward, terminated, truncated, extras = env.step(action)
    total_reward_history.append(reward.mean().item())

plt.figure(figsize=(6, 3))
plt.plot(total_reward_history)
plt.xlabel("step")
plt.ylabel("mean total reward (across envs)")
plt.title("Total reward, random actions")
plt.show()

## 5. Reward functions — the part you actually want to tweak

Every reward term is a plain Python function `(env, **params) -> Tensor[num_envs]`,
registered by name with a weight in `_build_rewards()`:

| term | weight | what it does |
|---|---|---|
| [`reach_distance_reward`](ur3e_reach_env_cfg.py#L212) | `1.0` | negative distance from end-effector to target |
| [`success_bonus`](ur3e_reach_env_cfg.py#L220) | `1.0` | +1/step while within 3cm of target |
| [`settle_penalty`](ur3e_reach_env_cfg.py#L237) | `-0.5` | penalizes joint velocity, but *only* once close to the target |
| `mdp_rewards.action_rate_l2` | `-0.01` | built-in mjlab term; penalizes jerky actions |
| [`ee_speed_limit_penalty`](ur3e_reach_env_cfg.py#L266) | `-2.0` | hinge penalty above 0.12 m/s end-effector speed |
| [`joint_vel_limit_penalty`](ur3e_reach_env_cfg.py#L299) | `-1.5` | per-joint hinge penalty above 25% of the real UR3e's rated velocity |
| `mdp_rewards.joint_vel_l2` | `-0.001` | general smoothness regularizer |
| [`collision_cost`](ur3e_reach_env_cfg.py#L316) ×2 | `-1.0` each | self-collision / table-collision, from contact sensors |

All of them are assembled in one place:
**[`_build_rewards()` at `ur3e_reach_env_cfg.py:446`](ur3e_reach_env_cfg.py#L446)**.
That function is the single place to edit for a permanent change — add a dict entry,
change a `weight=`, or point `func=` at a new function.

The cells below do the same thing *live*, without touching the file, so you can see
the effect before committing anything to it.

In [ ]:
print(f"{'term':<18}{'weight':>8}   params")
for name in env.reward_manager.active_terms:
    term_cfg = env.reward_manager.get_term_cfg(name)
    print(f"{name:<18}{term_cfg.weight:>8}   {term_cfg.params}")

### A rollout helper that reports the per-term breakdown

`env.reward_manager.get_active_iterable_terms(env_idx)` returns each term's
**already-weighted** contribution to the reward, for one env, at the current step --
exactly what you want in order to see which term dominates at any given moment.

In [ ]:
from mjlab.managers import RewardTermCfg
from ur3e.ur3e_reach_env_cfg import get_ur3e_reach_env_cfg


def make_ur3e_env(num_envs=4, reward_overrides=None, disable_terms=None,
                   extra_terms=None, **cfg_kwargs):
    """Build a fresh UR3e reach env, with optional reward tweaks layered on top
    of the file's defaults -- no need to edit ur3e_reach_env_cfg.py just to try
    something out.

    reward_overrides: {"term_name": new_weight}
    disable_terms:    ["term_name", ...]              -- fully removes the term
    extra_terms:      {"term_name": RewardTermCfg(...)} -- adds a new term
    """
    cfg = get_ur3e_reach_env_cfg(**cfg_kwargs)
    cfg.scene.num_envs = num_envs
    for name, weight in (reward_overrides or {}).items():
        cfg.rewards[name].weight = weight
    for name in (disable_terms or []):
        cfg.rewards.pop(name, None)
    cfg.rewards.update(extra_terms or {})
    return ManagerBasedRlEnv(cfg, device="cpu")


def rollout_with_reward_breakdown(env, num_steps=150, env_idx=0, action_scale=0.3):
    action_dim = env.action_manager.total_action_dim
    history = {name: [] for name in env.reward_manager.active_terms}
    distance_history = []
    reach_cmd = env.command_manager.get_term("reach_target")

    env.reset()
    for _ in range(num_steps):
        action = torch.randn(env.num_envs, action_dim) * action_scale
        env.step(action)
        for name, values in env.reward_manager.get_active_iterable_terms(env_idx):
            history[name].append(values[0])
        distance_history.append(reach_cmd.metrics["position_error"][env_idx].item())

    return history, distance_history


def plot_reward_breakdown(history, distance_history, title=""):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    for name, values in history.items():
        ax1.plot(values, label=name)
    ax1.set_ylabel("per-step reward contribution")
    ax1.legend(loc="upper right", fontsize=8, ncol=2)
    ax1.set_title(title or "Reward breakdown")

    ax2.plot(distance_history, color="black")
    ax2.axhline(0.03, color="green", linestyle="--", label="success threshold")
    ax2.set_ylabel("distance to target (m)")
    ax2.set_xlabel("step")
    ax2.legend()
    plt.tight_layout()
    plt.show()

### Baseline (no overrides)

In [ ]:
baseline_env = make_ur3e_env(num_envs=4)
history, distances = rollout_with_reward_breakdown(baseline_env)
plot_reward_breakdown(history, distances, title="Baseline reward weights")
baseline_env.close()

### Try it: change a weight

`reward_overrides` patches the config dict right before the env is built --
`ur3e_reach_env_cfg.py` is untouched.

In [ ]:
tuned_env = make_ur3e_env(
    num_envs=4,
    reward_overrides={
        "ee_speed_limit": -5.0,   # was -2.0 -- punish overspeed harder
        "success_bonus": 2.0,     # was 1.0  -- reward reaching more
    },
)
history, distances = rollout_with_reward_breakdown(tuned_env)
plot_reward_breakdown(history, distances, title="ee_speed_limit=-5.0, success_bonus=2.0")
tuned_env.close()

### Try it: add a brand-new reward term

Any reward function has the signature `(env, **params) -> Tensor[num_envs]`. Define
one inline and hand it to `extra_terms` as a `RewardTermCfg` -- no file edit needed
until you're happy with it and want to make it permanent.

In [ ]:
def elbow_bend_bonus(env, entity_name="ur3e", joint_name="elbow_joint", target_angle=-1.2):
    """Toy example: reward keeping the elbow near a preferred bend angle."""
    arm = env.scene[entity_name]
    idx = arm.joint_names.index(joint_name)
    joint_angle = arm.data.joint_pos[:, idx]
    return -torch.square(joint_angle - target_angle)


custom_env = make_ur3e_env(
    num_envs=4,
    extra_terms={
        "elbow_bend_bonus": RewardTermCfg(
            func=elbow_bend_bonus,
            weight=0.2,
            params={"joint_name": "elbow_joint", "target_angle": -1.2},
        ),
    },
)
print("reward terms now:", custom_env.reward_manager.active_terms)
history, distances = rollout_with_reward_breakdown(custom_env)
plot_reward_breakdown(history, distances, title="With custom elbow_bend_bonus term")
custom_env.close()

### Making a change permanent

Once you've found overrides you like, copy them into
[`_build_rewards()` in `ur3e_reach_env_cfg.py`](ur3e_reach_env_cfg.py#L446) (either
change an existing `weight=`, or add a new `RewardTermCfg` entry to the `rewards`
dict, same as `extra_terms` above). Restart this notebook's kernel (or
`importlib.reload` the module) afterwards so the registered task config picks up the
edit -- `import ur3e` only registers the task once per kernel session.

## 6. Training for real

This notebook's copy of the task (under `MyCode/ur3e/`) is a standalone package --
it's not wired into any project's CLI entry points, so `uv run train` won't see
`Mjlab-UR3e-Reach` from here, and `MyCode/.venv` has no `pyproject.toml` of its own
to run `uv run` against in the first place.

The actual training/play CLI lives in **`my_mjlab_project`**
(`MyCode/hackathon/my_mjlab_project`), which has `mjlab`'s `train`/`play` scripts and
an RSL-RL PPO runner config already set up for this task
([`__init__.py`](../hackathon/my_mjlab_project/src/my_mjlab_project/__init__.py) in
that project registers it). From that project's directory:

```bash
uv sync                                       # recreate its own .venv first (removed --
                                               # this notebook now runs standalone on
                                               # MyCode/.venv instead)
uv run train --task Mjlab-UR3e-Reach          # headless training
uv run play --task Mjlab-UR3e-Reach --agent trained   # watch a trained checkpoint
```

One more thing to know before that works: that project's top-level `__init__.py`
currently has `#import my_mjlab_project.tasks.ur3e_reach` **commented out** (it's
focused on the `trailer_park` task right now) -- uncomment it to register the UR3e
task there too.

Any reward changes you've validated here need to be copied into that project's copy
of `ur3e_reach_env_cfg.py` too, since training reads from there, not from this
notebook's copy.

## 7. Cleanup

In [ ]:
env.close()
render_env.close()